# ALBERT (A Lite BERT) Overview

## Key Features and Improvements

### 1. Factorized Embedding Parameterization

In BERT, the embedding layer maps input tokens to dense vectors. If the vocabulary size is $V$ and the hidden size is $H$, the embedding matrix has dimensions $V \times H$. This can result in a very large number of parameters, especially if $V$ and $H$ are large [1].

ALBERT reduces the number of parameters in the embedding layer by factorizing the embedding matrix into two smaller matrices. Let's denote:

- $E$ as the embedding size (typically much smaller than $H$)
- $V$ as the vocabulary size
- $H$ as the hidden size

Instead of having a single $V \times H$ matrix, ALBERT uses two matrices:

- $V \times E$: maps the input tokens to an intermediate size $E$
- $E \times H$: maps the intermediate vectors to the final hidden size $H$

Mathematically, the factorized embedding process is as follows:

1. **Initial Embedding**: $\mathbf{E}_{init} \in \mathbb{R}^{V \times E}$
2. **Intermediate Embedding to Hidden**: $\mathbf{E}_{hidden} \in \mathbb{R}^{E \times H}$

The combined embedding for a token $t$ is given by:

$\mathbf{E}(t) = \mathbf{E}_{init}(t) \cdot \mathbf{E}_{hidden}$

Where $\mathbf{E}_{init}(t)$ is the embedding of token $t$ from the $V \times E$ matrix [2].

### 2. Cross-Layer Parameter Sharing

In BERT, each transformer layer has its own set of parameters, including weights for self-attention and feed-forward networks. This results in a large number of parameters as the number of layers increases [1].

ALBERT shares parameters across layers, reducing the total number of unique parameters. This can be implemented in two ways:

1. **All Layers Share Parameters**: All layers use the same set of parameters. If there are $L$ layers, they all share the same weights $\mathbf{W}_{attn}$ for attention and $\mathbf{W}_{ffn}$ for the feed-forward network.
   
2. **Parameter Sharing by Groups**: Layers are divided into groups, with each group sharing parameters. For example, if there are 12 layers, they might be grouped into 3 groups of 4 layers each, where layers within each group share parameters [2].

Mathematically, if $\mathbf{W}_{attn}$ and $\mathbf{W}_{ffn}$ are the attention and feed-forward parameters for layer $l$, then for all layers $l$:

$\mathbf{W}_{attn}^l = \mathbf{W}_{attn}$
$\mathbf{W}_{ffn}^l = \mathbf{W}_{ffn}$

### 3. Sentence Order Prediction (SOP)

In BERT, the Next Sentence Prediction (NSP) task helps the model understand relationships between sentences. However, this task is relatively easy and might not capture more nuanced sentence relationships [1].

ALBERT introduces the Sentence Order Prediction (SOP) task, where given two consecutive segments $A$ and $B$ from a document:

- The model predicts whether $B$ follows $A$ (as in the original order) or if $B$ precedes $A$ (swapped order).

The SOP task is formulated as a binary classification problem. For each input pair $(A, B)$:

- If $B$ follows $A$ (correct order), the label is 0.
- If $B$ precedes $A$ (swapped order), the label is 1.

The objective is to minimize the cross-entropy loss for this classification task [2].

### 4. Overall Parameter Reduction

Combining factorized embedding parameterization and cross-layer parameter sharing, the total number of parameters in ALBERT can be significantly reduced compared to BERT [2].

- **Factorized Embedding Parameterization**: Reduces the number of parameters from $V \times H$ to $V \times E + E \times H$.
- **Cross-Layer Parameter Sharing**: Reduces the number of parameters in the transformer layers by reusing the same parameters across multiple layers.

### Summary

To summarize, ALBERT achieves efficiency and maintains performance by:

1. **Reducing Embedding Parameters**: Factorizing the embedding matrix into two smaller matrices.
2. **Sharing Parameters Across Layers**: Reusing the same parameters across multiple transformer layers.
3. **Improving Sentence Understanding**: Introducing the SOP task to better capture sentence coherence and order.
4. **Maintaining Performance**: Despite having fewer parameters, ALBERT achieves similar or better performance on NLP tasks compared to BERT [2].

These optimizations make ALBERT a more efficient model in terms of memory usage and training time while preserving the strong performance characteristics of BERT [1][2].

References:

[1] Devlin, J., Chang, M. W., Lee, K., & Toutanova, K. (2018). BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding. arXiv preprint arXiv:1810.04805.

[2] Lan, Z., Chen, M., Goodman, S., Gimpel, K., Sharma, P., & Soricut, R. (2019). ALBERT: A Lite BERT for Self-supervised Learning of Language Representations. arXiv preprint arXiv:1909.11942.

In [2]:
import sys
import torch

print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")
!pip install torch==2.3.1+cu121 transformers==4.38.2 datasets==2.17.1

Python version: 3.10.12 (main, Mar 22 2024, 16:50:05) [GCC 11.4.0]
PyTorch version: 2.3.1+cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.7/130.7 kB 1.2 MB/s eta 0:00:00
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cusparse_cu12-12.1.0.106-py3-none-manylinux1_

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import math
import random

# Check versions
import sys
print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Simplified ALBERT Configuration
class AlbertConfig:
    def __init__(self, vocab_size=30000, hidden_size=128, num_hidden_layers=2,
                 num_attention_heads=4, intermediate_size=512, max_position_embeddings=512):
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.num_hidden_layers = num_hidden_layers
        self.num_attention_heads = num_attention_heads
        self.intermediate_size = intermediate_size
        self.max_position_embeddings = max_position_embeddings

# Simplified ALBERT Model
class AlbertModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config

        # Factorized Embedding
        self.word_embeddings = nn.Embedding(config.vocab_size, config.hidden_size // 4)
        self.word_embeddings_projector = nn.Linear(config.hidden_size // 4, config.hidden_size)

        # Position Embeddings
        self.position_embeddings = nn.Embedding(config.max_position_embeddings, config.hidden_size)

        # Layer Norm
        self.layer_norm = nn.LayerNorm(config.hidden_size)

        # Transformer Layers
        self.layers = nn.ModuleList([TransformerLayer(config) for _ in range(config.num_hidden_layers)])

        # MLM head
        self.mlm_head = nn.Linear(config.hidden_size, config.vocab_size)

        # SOP head
        self.sop_head = nn.Linear(config.hidden_size, 2)

    def forward(self, input_ids, attention_mask=None):
        # Word embeddings
        embeddings = self.word_embeddings(input_ids)
        embeddings = self.word_embeddings_projector(embeddings)

        # Position embeddings
        position_ids = torch.arange(input_ids.size(1), dtype=torch.long, device=input_ids.device)
        position_ids = position_ids.unsqueeze(0).expand_as(input_ids)
        position_embeddings = self.position_embeddings(position_ids)

        # Combine embeddings
        embeddings = embeddings + position_embeddings
        embeddings = self.layer_norm(embeddings)

        # Apply Transformer layers
        hidden_states = embeddings
        for layer in self.layers:
            hidden_states = layer(hidden_states, attention_mask)

        return hidden_states

    def mlm_output(self, hidden_states):
        return self.mlm_head(hidden_states)

    def sop_output(self, hidden_states):
        return self.sop_head(hidden_states[:, 0, :])  # Use [CLS] token for SOP

class TransformerLayer(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.attention = MultiHeadAttention(config)
        self.intermediate = nn.Linear(config.hidden_size, config.intermediate_size)
        self.output = nn.Linear(config.intermediate_size, config.hidden_size)
        self.layer_norm1 = nn.LayerNorm(config.hidden_size)
        self.layer_norm2 = nn.LayerNorm(config.hidden_size)
        self.activation = nn.GELU()

    def forward(self, hidden_states, attention_mask=None):
        attention_output = self.attention(hidden_states, attention_mask)
        attention_output = self.layer_norm1(hidden_states + attention_output)

        intermediate_output = self.intermediate(attention_output)
        intermediate_output = self.activation(intermediate_output)
        layer_output = self.output(intermediate_output)
        layer_output = self.layer_norm2(attention_output + layer_output)

        return layer_output

class MultiHeadAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.num_attention_heads = config.num_attention_heads
        self.attention_head_size = config.hidden_size // config.num_attention_heads
        self.all_head_size = self.num_attention_heads * self.attention_head_size

        self.query = nn.Linear(config.hidden_size, self.all_head_size)
        self.key = nn.Linear(config.hidden_size, self.all_head_size)
        self.value = nn.Linear(config.hidden_size, self.all_head_size)

        self.dropout = nn.Dropout(0.1)

    def transpose_for_scores(self, x):
        new_x_shape = x.size()[:-1] + (self.num_attention_heads, self.attention_head_size)
        x = x.view(*new_x_shape)
        return x.permute(0, 2, 1, 3)

    def forward(self, hidden_states, attention_mask=None):
        query_layer = self.transpose_for_scores(self.query(hidden_states))
        key_layer = self.transpose_for_scores(self.key(hidden_states))
        value_layer = self.transpose_for_scores(self.value(hidden_states))

        attention_scores = torch.matmul(query_layer, key_layer.transpose(-1, -2))
        attention_scores = attention_scores / math.sqrt(self.attention_head_size)

        if attention_mask is not None:
            attention_scores = attention_scores + attention_mask

        attention_probs = nn.Softmax(dim=-1)(attention_scores)
        attention_probs = self.dropout(attention_probs)

        context_layer = torch.matmul(attention_probs, value_layer)
        context_layer = context_layer.permute(0, 2, 1, 3).contiguous()
        new_context_layer_shape = context_layer.size()[:-2] + (self.all_head_size,)
        context_layer = context_layer.view(*new_context_layer_shape)

        return context_layer

# Dummy dataset with MLM and SOP tasks
class DummyDataset(Dataset):
    def __init__(self, size=1000, seq_len=128, vocab_size=30000, mlm_probability=0.15):
        self.data = torch.randint(0, vocab_size, (size, seq_len))
        self.mlm_probability = mlm_probability
        self.vocab_size = vocab_size

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        tokens = self.data[idx].clone()

        # MLM task
        mlm_labels = tokens.clone()
        probability_matrix = torch.full(tokens.shape, self.mlm_probability)
        masked_indices = torch.bernoulli(probability_matrix).bool()
        mlm_labels[~masked_indices] = -100  # We only compute loss on masked tokens

        # 80% of the time, we replace masked input tokens with tokenizer.mask_token ([MASK])
        indices_replaced = torch.bernoulli(torch.full(tokens.shape, 0.8)).bool() & masked_indices
        tokens[indices_replaced] = self.vocab_size - 1  # Assume [MASK] token is the last token in vocab

        # 10% of the time, we replace masked input tokens with random word
        indices_random = torch.bernoulli(torch.full(tokens.shape, 0.5)).bool() & masked_indices & ~indices_replaced
        random_words = torch.randint(self.vocab_size, tokens.shape, dtype=torch.long)
        tokens[indices_random] = random_words[indices_random]

        # SOP task
        # Randomly decide if we swap sentences (50% probability)
        is_next = random.random() > 0.5
        if is_next:
            sop_label = 1
        else:
            sop_label = 0
            # Swap the halves of the sequence
            half = len(tokens) // 2
            tokens[:half], tokens[half:] = tokens[half:], tokens[:half]

        return tokens, mlm_labels, torch.tensor(sop_label)

# Initialize model and dataset
config = AlbertConfig()
model = AlbertModel(config).to(device)
dataset = DummyDataset()
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

# Loss and optimizer
mlm_criterion = nn.CrossEntropyLoss()
sop_criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# Training loop
num_epochs = 3
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for batch, mlm_labels, sop_labels in dataloader:
        batch, mlm_labels, sop_labels = batch.to(device), mlm_labels.to(device), sop_labels.to(device)

        optimizer.zero_grad()
        hidden_states = model(batch)

        # MLM loss
        mlm_logits = model.mlm_output(hidden_states)
        mlm_loss = mlm_criterion(mlm_logits.view(-1, config.vocab_size), mlm_labels.view(-1))

        # SOP loss
        sop_logits = model.sop_output(hidden_states)
        sop_loss = sop_criterion(sop_logits, sop_labels)

        # Combined loss
        loss = mlm_loss + sop_loss

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(dataloader)
    print(f"Epoch {epoch+1}/{num_epochs}, Average Loss: {avg_loss:.4f}")

print("Training complete!")

# Save the model
torch.save(model.state_dict(), 'simplified_albert_model_with_mlm_sop.pth')
print("Model saved as 'simplified_albert_model_with_mlm_sop.pth'")

Python version: 3.10.12 (main, Mar 22 2024, 16:50:05) [GCC 11.4.0]
PyTorch version: 2.3.1+cu121
Using device: cpu
Epoch 1/3, Average Loss: 11.1965
Epoch 2/3, Average Loss: 11.1528
Epoch 3/3, Average Loss: 11.1382
Training complete!
Model saved as 'simplified_albert_model_with_mlm_sop.pth'
